In [ ]:
#requirements
!pip install pygithub
!pip install gitpython
!pip install transformers[torch]
!pip install accelerate -U
!pip install datasets

from transformers import GPT2Config,GPT2LMHeadModel,GPT2Tokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
import os
import torch
import accelerate
from tokenizers import ByteLevelBPETokenizer
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)
torch.cuda.get_device_name(0)

In [ ]:
import time
from datetime import datetime
end_time=time.time()- (7*86400)
start_time=end_time-(8*86400)
import os
from github import Github
import git


#Data collection

In [ ]:

act=open("gitauth.txt","r").read()
g= Github(act)
print(g.get_user())


AuthenticatedUser(login=None)


In [ ]:
for i in range(1,90,2):
  start_time_str=datetime.utcfromtimestamp(start_time).strftime('%Y-%m-%d')
  end_time_str=datetime.utcfromtimestamp(end_time).strftime('%Y-%m-%d')

  query=f"pandas language:python created:{start_time_str}..{end_time_str}"
  print(query)

  end_time-= (i*86400) - 86400
  start_time -= (i*86400) -86400
  result= g.search_repositories(query)
  print(result.totalCount)

  # %rm -rf code_gen

  for repository in result:
    # os.system(f"git clone {repository.clone_url}code_gen")
    # git.Repo.clone_from(repository.clone_url, "/content/code_gen/{repository.owner.login}/{repository.name}")

    try:
      repository_url = repository.clone_url
      target_path = f"/content/code_gen/{repository.owner.login}/{repository.name}"
      git.Repo.clone_from(repository_url, target_path)
      print("Repository cloned successfully!")
    except git.exc.GitCommandError as e:
      print(f"Error cloning repository: {e}")
    break

In [ ]:
#  %rm -rf code_gen

# Data cleaning

In [ ]:
#data cleaning

for dirpath,dirnames,filenames in os.walk("/content/code_gen"):
  for f in filenames:
    fullpath=os.path.join(dirpath,f)
    # print(fullpath)

    if fullpath.endswith(".py"):
      # print("green: "+ fullpath)
      print(fullpath)
    else:
      # print("red: " + fullpath)
      os.remove(fullpath)



# Data Preprocessing

In [ ]:
newlinechar="<N>"

full_paths=[]
for dirpath,dirnames,filenames in os.walk("/content/code_gen"):
  for f in filenames:
    full_path=os.path.join(dirpath,f)
    full_paths.append(full_path)
print(len(full_paths))


with open("codegen_text.txt","a",encoding='utf-8') as f:
  max_char_length=1024
  min_char_length=150

  try:
    for fpath in full_paths:
      d=open(fpath,"r",encoding='utf-8').read()
      fd=d.replace("\n",newlinechar)

      if 100<len(d)<=max_char_length:
        # print(d)
        # print(len(d))
        f.write(fd+'\n')

      else:
        sd=fd.split(f"{newlinechar}{newlinechar}")
        substr=""
        for split in sd:
          substr+=split+f"{newlinechar}{newlinechar}"
          if min_char_length<=len(substr)<=max_char_length:
            f.write(substr+'\n')
            substr=""
  except Exception as e:
    print(str(e))


2884


# Tokenizaton

In [ ]:
paths=["codegen_text.txt"]

In [ ]:
tokenizer = ByteLevelBPETokenizer()

tokenizer.train(files=paths,vocab_size=52000,min_frequency=2,special_tokens=[
    "<s>",
    "</s>",
    "<pad>",
    "<unk>",
    "<mask>",
])

#save files to disk, manually create tokenizers folder
tokenizer.save_model("/content/tokenizers")

inp='print("hello world")'

t=tokenizer.encode(inp)

print(t.ids)
print(t.tokens)



[1312, 449, 11224, 12293, 985]
['print', '("', 'hello', 'Ġworld', '")']


In [ ]:


tokenizer= GPT2Tokenizer.from_pretrained("tokenizers")
tokenizer.add_special_tokens({
    "eos_token": "</s>",
    "bos_token": "<s>",
    "pad_token": "<pad>",
    "mask_token": "<mask>",
})

inp='print("hello world")'
t=tokenizer.encode(inp)
print(t)
print(tokenizer.decode(t))

[1312, 449, 11224, 12293, 985]
print("hello world")


# Training


In [ ]:
#now config

config=GPT2Config(
    vocab_size=tokenizer.vocab_size,
    bos_token=tokenizer.bos_token_id,
    eos_token=tokenizer.eos_token_id
)

model=GPT2LMHeadModel(config)

paths=["codegen_text.txt"]

# dataset= load_dataset("text", datafiles=paths)
dataset = load_dataset("text", data_files=paths)

def encode(lines):
  return tokenizer(lines['text'], add_special_tokens=True,truncation=True,max_length=512)


dataset.set_transform(encode)
dataset = dataset['train']

data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True,mlm_probability=0.15)



Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# for managing cupa resources

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# torch.cuda.empty_cache()

# %rm -rf CodeGen_gpt2

# import gc
# torch.cuda.empty_cache()
# gc.collect()

# import os
# import gc
# gc.collect()
# torch.cuda.empty_cache()

# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
#training on limited resources, thus the values are low for getting acceptable results

training_args = TrainingArguments(
    output_dir="./CodeGen_gpt2",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=3,
    save_total_limit=2,
    prediction_loss_only=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()
# trainer.save_model("CodeGen_gpt2")

Step,Training Loss
500,5.831900
1000,5.106500
1500,4.947400
2000,4.833000
2500,4.677100
3000,4.606500
3500,4.507800
4000,4.470700
4500,4.436000
5000,4.427300


TrainOutput(global_step=7744, training_loss=4.596264326867979, metrics={'train_runtime': 1302.5992, 'train_samples_per_second': 17.835, 'train_steps_per_second': 5.945, 'total_flos': 1653589686528000.0, 'train_loss': 4.596264326867979, 'epoch': 1.0})

# Saving to Hugging Face

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
pm= "adikrakshak/gpt2-experiment"

model.push_to_hub(
    pm,use_auth_token=True
)

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:834: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/adikrakshak/gpt2-experiment/commit/4393aba23b56f4f41daf959d4b3b0386efb54b96', commit_message='Upload model', commit_description='', oid='4393aba23b56f4f41daf959d4b3b0386efb54b96', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
# !python --version
# !pip show accelerate
!nvidia-smi

Mon May  6 22:38:11 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   69C    P0              31W /  70W |  14961MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:

model=GPT2LMHeadModel.from_pretrained("adikrakshak/gpt2-experiment").to("cuda")

# while True:
  # inp=input(">>> ")

input_ids= tokenizer.encode(inp, return_tensors="pt").to("cuda")
beam_output= model.generate(
    input_ids,
    max_length=300,
    num_beams=5,
    temperature=0.7,
    no_repeat_ngram_size=5,
    num_return_sequences=2,
    # return_dict_in_generate= True,
    output_scores=True
)

for beam in beam_output:
  out=tokenizer.decode(beam)
  fout=out.replace("<N>","\n")

    # print(str(fout))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


#Downloading/Saving models and files locally

In [ ]:
# from google.colab import files

# Zip each folder and file individually
# !zip -r /content/tokenizers.zip /content/tokenizers
# !zip -r /content/code_gen.zip /content/code_gen
# !zip -r /content/CodeGen_gpt22.zip /content/CodeGen_gpt2
# !zip /content/codegen_text.zip /content/codegen_text.txt

# # Download the zip files
# files.download("/content/tokenizers.zip")
# files.download("/content/code_gen.zip")
# files.download("/content/CodeGen_gpt22.zip")
# files.download("/content/codegen_text.zip")



  adding: content/CodeGen_gpt2/ (stored 0%)
  adding: content/CodeGen_gpt2/training_args.bin (deflated 51%)
  adding: content/CodeGen_gpt2/model.safetensors (deflated 7%)
  adding: content/CodeGen_gpt2/generation_config.json (deflated 24%)
  adding: content/CodeGen_gpt2/checkpoint-4100/ (stored 0%)
  adding: content/CodeGen_gpt2/checkpoint-4100/training_args.bin (deflated 51%)
  adding: content/CodeGen_gpt2/checkpoint-4100/scheduler.pt (deflated 55%)
  adding: content/CodeGen_gpt2/checkpoint-4100/trainer_state.json (deflated 68%)
  adding: content/CodeGen_gpt2/checkpoint-4100/rng_state.pth (deflated 25%)
  adding: content/CodeGen_gpt2/checkpoint-4100/model.safetensors (deflated 7%)
  adding: content/CodeGen_gpt2/checkpoint-4100/generation_config.json (deflated 24%)
  adding: content/CodeGen_gpt2/checkpoint-4100/optimizer.pt (deflated 9%)
  adding: content/CodeGen_gpt2/checkpoint-4100/config.json (deflated 52%)
  adding: content/CodeGen_gpt2/runs/ (stored 0%)
  adding: content/CodeGen_g

In [ ]:
newlinechar="<N>"
def encode_newlines(inp):
  return inp.replace("\n",newlinechar)

def decode_newline(inp):
  return inp.replace(newlinechar,"\n")

inp=input("\n>>>")

input_ids= tokenizer.encode(inp, return_tensors="pt").to("cuda")
beam_output= model.generate(
    input_ids,
    max_length=100,
    num_beams=3,
    temperature=0.7,
    no_repeat_ngram_size=5,
    num_return_sequences=3,
    # return_dict_in_generate= True,
    # output_scores=True,
)

for beam in beam_output:
  out=tokenizer.decode(beam)
  fout=out.replace("<N>","\n")

  print(str(fout))

# for out in beam_output['sequences']:
#   print (tokenizer.decode(out))

In [ ]:
newlinechar="<N>"
def encode_newline(inp):
  return inp.replace("\n",newlinechar)

def decode_newline(inp):
  return inp.replace(newlinechar,"\n")
def com(inp):
  inp=encode_newline(inp)
  newline_count=inp.count(newlinechar)

  input_ids= tokenizer.encode(inp, return_tensors="pt").to("cuda")
  beam_output= model.generate(
      input_ids,
      max_length=100,
      num_beams=5, #for diverse results
      temperature=0.7,
      no_repeat_ngram_size=5,
      num_return_sequences=3,
      return_dict_in_generate= True,
      output_scores=True
  )
  cout=beam_output['sequences'][0]
  print(15*"-")
  print (decode_newline(tokenizer.decode(cout)))
  print(15*"-")
  autocom=""
  split=decode_newline(tokenizer.decode(cout)).split('\n')
  for s in split[:newline_count+1]:
    autocom+=s+'\n'
  for seqscore in beam_output['sequences_scores']:
    print(seqscore)
  return autocom

In [ ]:
while True:
  inp=input("\,>>>>")
  msg= com(inp)
  print(msg)